In [0]:
dbutils.widgets.removeAll()

In [0]:
from datetime import datetime, timezone

dbutils.widgets.text(
    "environment",
    "dev",
    "Environment"
)

dbutils.widgets.text(
    "ingestion_timestamp",
    datetime.now(timezone.utc).isoformat(),
    "Ingestion Timestamp"
)

environment = dbutils.widgets.get("environment").lower()

ingestion_timestamp = (
    dbutils.widgets.get("ingestion_timestamp").strip()
)

if environment not in ["dev", "prod"]:
    raise ValueError(
        "Environment must be either 'dev' or 'prod'."
    )

config = {
    "dev": {
        "source_catalog": "fc_saleslt_dev",
        "target_catalog": "saleslt_dev",
        "storage_account": "stcentralusjrdev"
    },
    "prod": {
        "source_catalog": "fc_saleslt_prod",
        "target_catalog": "saleslt_prod",
        "storage_account": "stcentralusjrprod"
    }
}

env = config[environment]

source_catalog = env["source_catalog"]
target_catalog = env["target_catalog"]
storage_account = env["storage_account"]

print("=" * 60)
print("SALESLT - BRONZE INGESTION")
print("=" * 60)
print(f"Environment           : {environment}")
print(f"Ingestion timestamp   : {ingestion_timestamp}")
print(f"Source catalog        : {source_catalog}")
print(f"Target catalog        : {target_catalog}")
print(f"Storage account       : {storage_account}")
print("=" * 60)

In [0]:
table_config = {
    "customer": {
        "source_table": f"{source_catalog}.SalesLT.Customer",
        "target_table": f"{target_catalog}.bronze.customer_raw",
        "target_path": (
            f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
            "saleslt/bronze/customer_raw/"
        ),
        "merge_key": "CustomerID"
    },

    "product": {
        "source_table": f"{source_catalog}.SalesLT.Product",
        "target_table": f"{target_catalog}.bronze.product_raw",
        "target_path": (
            f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
            "saleslt/bronze/product_raw/"
        ),
        "merge_key": "ProductID"
    },

    "product_category": {
        "source_table": f"{source_catalog}.SalesLT.ProductCategory",
        "target_table": f"{target_catalog}.bronze.product_category_raw",
        "target_path": (
            f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
            "saleslt/bronze/product_category_raw/"
        ),
        "merge_key": "ProductCategoryID"
    },

    "sales_order_header": {
        "source_table": f"{source_catalog}.SalesLT.SalesOrderHeader",
        "target_table": f"{target_catalog}.bronze.sales_order_header_raw",
        "target_path": (
            f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
            "saleslt/bronze/sales_order_header_raw/"
        ),
        "merge_key": "SalesOrderID"
    },

    "sales_order_detail": {
        "source_table": f"{source_catalog}.SalesLT.SalesOrderDetail",
        "target_table": f"{target_catalog}.bronze.sales_order_detail_raw",
        "target_path": (
            f"abfss://lakehouse@{storage_account}.dfs.core.windows.net/"
            "saleslt/bronze/sales_order_detail_raw/"
        ),
        "merge_key": "SalesOrderDetailID"
    }
}

In [0]:
from pyspark.sql.functions import lit

source_dfs = {}

for table_name, cfg in table_config.items():

    print(f"Reading source: {cfg['source_table']}")

    df = spark.sql(f"""
        SELECT *
        FROM {cfg['source_table']}
    """)

    df = (
        df.withColumn(
            "ingestion_timestamp",
            lit(ingestion_timestamp).cast("timestamp")
        )
    )

    source_dfs[table_name] = df

    print(
        f"{table_name}: {df.count()} records"
    )

In [0]:
from delta.tables import DeltaTable


def merge_bronze_snapshot(
    source_df,
    target_table,
    target_path,
    merge_key
):
    """
    Creates an external Delta Bronze table on the initial load.

    Subsequent executions synchronize the Bronze table with the
    current Lakehouse Federation source snapshot.
    """

    if not spark.catalog.tableExists(target_table):

        print(
            f"Initial load. Creating Bronze table: "
            f"{target_table}"
        )

        (
            source_df.write
                .format("delta")
                .mode("append")
                .option("path", target_path)
                .saveAsTable(target_table)
        )

    else:

        print(
            f"Synchronizing Bronze table: "
            f"{target_table}"
        )

        target = DeltaTable.forName(
            spark,
            target_table
        )

        (
            target.alias("target")

                .merge(
                    source_df.alias("source"),
                    f"target.{merge_key} = source.{merge_key}"
                )

                .whenMatchedUpdateAll()

                .whenNotMatchedInsertAll()

                .whenNotMatchedBySourceDelete()

                .execute()
        )

    print(
        f"Bronze synchronization completed: "
        f"{target_table}"
    )

In [0]:
for table_name, cfg in table_config.items():

    print("\n" + "=" * 60)
    print(f"Processing: {table_name}")
    print("=" * 60)

    merge_bronze_snapshot(
        source_df=source_dfs[table_name],
        target_table=cfg["target_table"],
        target_path=cfg["target_path"],
        merge_key=cfg["merge_key"]
    )